In [ ]:
import time
import numpy as np
from torch_geometric.data import HeteroData

# Import your model implementations
# Assuming you have these files in your project:
# from han_cancer_classification import main as han_main
# from gat_cancer_classification import main as gat_main  
# from gcn_cancer_classification import main as gcn_main

def compare_all_models(hetero_data: HeteroData, max_path_length=3, top_k=10, verbose=True):
    """
    Compare HAN, GAT, and GCN models for cancer subtype classification
    
    Args:
        hetero_data: HeteroData object containing the graph
        max_path_length: Maximum length of meta-paths to explore
        top_k: Number of top results to display
        verbose: Whether to print detailed results
    
    Returns:
        dict: Comparison results for all models
    """
    
    if verbose:
        print("="*100)
        print("COMPREHENSIVE MODEL COMPARISON FOR CANCER SUBTYPE CLASSIFICATION")
        print("="*100)
        print(f"Dataset info:")
        print(f"  Node types: {list(hetero_data.node_types)}")
        print(f"  Edge types: {list(hetero_data.edge_types)}")
        print(f"  Target nodes (cases): {hetero_data['case'].num_nodes}")
        print(f"  Meta-path max length: {max_path_length}")
        print("="*100)
    
    results = {}
    
    # Test GCN (simplest model first)
    if verbose:
        print("\n🔹 TESTING GCN MODEL")
        print("-" * 50)
    
    try:
        start_time = time.time()
        # Replace this import with your actual GCN implementation
        from gcn_cancer_classification import main as gcn_main
        gcn_results = gcn_main(hetero_data, max_path_length, top_k)
        gcn_time = time.time() - start_time
        
        if gcn_results:
            results['GCN'] = {
                'best_accuracy': gcn_results['best_accuracy'],
                'best_metapath': gcn_results['best_path_string'],
                'total_time': gcn_time,
                'all_results': gcn_results['all_results']
            }
            if verbose:
                print(f"✅ GCN completed in {gcn_time:.2f}s")
                print(f"   Best accuracy: {gcn_results['best_accuracy']:.4f}")
        else:
            results['GCN'] = {'error': 'No results obtained'}
            if verbose:
                print("❌ GCN failed to produce results")
                
    except Exception as e:
        results['GCN'] = {'error': str(e)}
        if verbose:
            print(f"❌ GCN failed with error: {e}")
    
    # Test GAT
    if verbose:
        print("\n🔹 TESTING GAT MODEL")
        print("-" * 50)
    
    try:
        start_time = time.time()
        # Replace this import with your actual GAT implementation
        from gat_cancer_classification import main as gat_main
        gat_results = gat_main(hetero_data, max_path_length, top_k)
        gat_time = time.time() - start_time
        
        if gat_results:
            results['GAT'] = {
                'best_accuracy': gat_results['best_accuracy'],
                'best_metapath': gat_results['best_path_string'],
                'total_time': gat_time,
                'all_results': gat_results['all_results']
            }
            if verbose:
                print(f"✅ GAT completed in {gat_time:.2f}s")
                print(f"   Best accuracy: {gat_results['best_accuracy']:.4f}")
        else:
            results['GAT'] = {'error': 'No results obtained'}
            if verbose:
                print("❌ GAT failed to produce results")
                
    except Exception as e:
        results['GAT'] = {'error': str(e)}
        if verbose:
            print(f"❌ GAT failed with error: {e}")
    
    # Test HAN (most complex model last)
    if verbose:
        print("\n🔹 TESTING HAN MODEL")
        print("-" * 50)
    
    try:
        start_time = time.time()
        # Replace this import with your actual HAN implementation
        # Note: HAN has different parameters, adjust as needed
        # from han_cancer_classification import run_comprehensive_metapath_experiment as han_main
        # han_results = han_main(hetero_data, 'case', max_path_length, 20, 10, 2, 6)
        
        # For now, we'll skip HAN since it has a more complex interface
        # You can uncomment and adjust the above lines when ready
        
        results['HAN'] = {'error': 'HAN implementation needs to be integrated'}
        if verbose:
            print("⚠️  HAN implementation needs to be integrated with this comparison script")
            
    except Exception as e:
        results['HAN'] = {'error': str(e)}
        if verbose:
            print(f"❌ HAN failed with error: {e}")
    
    # Display comparison results
    if verbose:
        print("\n" + "="*100)
        print("📊 FINAL COMPARISON RESULTS")
        print("="*100)
        
        # Create comparison table
        successful_models = [model for model in results.keys() if 'best_accuracy' in results[model]]
        
        if successful_models:
            print(f"\n{'Model':<8} {'Best Accuracy':<15} {'Training Time':<15} {'Best Meta-path':<50}")
            print("-" * 100)
            
            # Sort by accuracy
            sorted_models = sorted(successful_models, 
                                 key=lambda x: results[x]['best_accuracy'], 
                                 reverse=True)
            
            for i, model in enumerate(sorted_models, 1):
                result = results[model]
                metapath = result['best_metapath']
                if len(metapath) > 45:
                    metapath = metapath[:42] + "..."
                
                print(f"{i}. {model:<5} {result['best_accuracy']:<15.4f} "
                      f"{result['total_time']:<15.2f} {metapath:<50}")
            
            # Winner announcement
            winner = sorted_models[0]
            print(f"\n🏆 WINNER: {winner} with accuracy {results[winner]['best_accuracy']:.4f}")
            
            # Accuracy differences
            if len(sorted_models) > 1:
                print(f"\n📈 Performance Gaps:")
                best_acc = results[sorted_models[0]]['best_accuracy']
                for model in sorted_models[1:]:
                    gap = best_acc - results[model]['best_accuracy']
                    print(f"   {sorted_models[0]} vs {model}: +{gap:.4f} accuracy points")
        
        else:
            print("❌ No models completed successfully!")
        
        # Show failed models
        failed_models = [model for model in results.keys() if 'error' in results[model]]
        if failed_models:
            print(f"\n⚠️  Failed Models:")
            for model in failed_models:
                print(f"   {model}: {results[model]['error']}")
    
    return results

def quick_comparison(hetero_data: HeteroData):
    """Quick comparison with default parameters"""
    return compare_all_models(hetero_data, max_path_length=2, top_k=5, verbose=True)

def detailed_comparison(hetero_data: HeteroData):
    """Detailed comparison with more extensive search"""
    return compare_all_models(hetero_data, max_path_length=3, top_k=10, verbose=True)

def save_comparison_results(results, filename="model_comparison_results.txt"):
    """Save comparison results to a file"""
    
    with open(filename, 'w') as f:
        f.write("Cancer Subtype Classification - Model Comparison Results\n")
        f.write("=" * 60 + "\n\n")
        
        successful_models = [model for model in results.keys() if 'best_accuracy' in results[model]]
        
        if successful_models:
            # Sort by accuracy
            sorted_models = sorted(successful_models, 
                                 key=lambda x: results[x]['best_accuracy'], 
                                 reverse=True)
            
            f.write(f"{'Rank':<6} {'Model':<8} {'Accuracy':<12} {'Time (s)':<10} Best Meta-path\n")
            f.write("-" * 80 + "\n")
            
            for i, model in enumerate(sorted_models, 1):
                result = results[model]
                f.write(f"{i:<6} {model:<8} {result['best_accuracy']:<12.4f} "
                       f"{result['total_time']:<10.2f} {result['best_metapath']}\n")
            
            f.write(f"\nWinner: {sorted_models[0]} with accuracy {results[sorted_models[0]]['best_accuracy']:.4f}\n")
        
        # Failed models
        failed_models = [model for model in results.keys() if 'error' in results[model]]
        if failed_models:
            f.write(f"\nFailed Models:\n")
            for model in failed_models:
                f.write(f"  {model}: {results[model]['error']}\n")
    
    print(f"Results saved to {filename}")


def run_comprehensive_test(hetero_data):
    """Run comprehensive test with detailed parameters"""
    print("🚀 Running comprehensive model comparison...")
    results = detailed_comparison(hetero_data)
    save_comparison_results(results, "comprehensive_comparison.txt")
    return results
